# Latente rum: hvordan en model gemmer "mening" som tal

Velkommen til forberedelsen til transformer-workshoppen!

I autoencoder-emnet så I, at et netværk kunne presse et billede ned til få tal — en **vektor** — og at
*ens* ting endte *tæt* på hinanden i det, vi kaldte det **latente rum**. En transformer (modellen bag fx
ChatGPT) bygger på præcis samme idé — nu bruger vi den på **tekst**: hvert tegn bliver lavet om til en
vektor, før modellen regner videre. I denne notebook bygger vi broen fra autoencoder til transformer og
kigger direkte ind i den færdige models latente rum.

> **Om opgaverne:** Nogle opgaver beder dig om at *tænke* over noget og tale med din sidemand — dem skal
> du ikke skrive svaret ned på. Opgaver mærket **(ekstra)** er samlet allersidst i notebooken, hvis du får
> tid og lyst til lidt mere.
>
> Noget af det her er nyt og kan føles udfordrende i starten — og det er helt okay. Vi forklarer hvert skridt så klart og tydeligt, vi kan, og der er et hint til hver opgave, hvis du går i stå. Tag dig endelig god tid.

## Setup

In [ ]:
# Henter den færdige transformer-model fra GitHub (Plan B: upload base_model.pth via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/27-Models/base_model.pth

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print("Klar!")

# 1: Fra autoencoder til latent rum

Husk autoencoderen: den pressede 784 pixels ned til fx 2 tal og kunne bygge billedet op igen. De 2 tal var
et **punkt** i et latent rum, og cifre der lignede hinanden, lå tæt på hinanden — helt af sig selv.

Den samme idé driver sprogmodeller: et ord (eller tegn) beskrives med en **vektor** (en liste af tal), og
"betydning" bliver til **placering** i rummet. For at få en god intuition starter vi med nogle vektorer, vi
selv finder på — så kan du se med det samme, hvordan afstand og retning kommer til at betyde noget.

In [ ]:
# Vi opdigter selv nogle 2D-"betydningsvektorer". (I virkeligheden LÆRER modellen dem under træning —
# her vælger vi dem i hånden, så vi kan se idéen tydeligt med det samme.)
word_vectors = {
    "konge":    torch.tensor([2.0, 3.0]),
    "dronning": torch.tensor([2.2, 2.8]),
    "prins":    torch.tensor([1.8, 2.6]),
    "hund":     torch.tensor([-2.0, -1.0]),
    "kat":      torch.tensor([-1.7, -1.3]),
    "bil":      torch.tensor([3.0, -3.0]),
}

plt.figure(figsize=(6, 5))
for word, v in word_vectors.items():
    plt.scatter(v[0], v[1])                         # ét punkt pr. ord
    plt.annotate(word, (v[0], v[1]), fontsize=12)   # skriv ordet lige ved punktet
plt.axhline(0, color="gray", lw=0.5)
plt.axvline(0, color="gray", lw=0.5)
plt.title("Ord som punkter i et (opdigtet) latent rum")
plt.show()

Læg mærke til: **konge**, **dronning** og **prins** klumper sammen (kongelige), mens **hund** og
**kat** ligger for sig (kæledyr). Retningen og afstanden bærer altså en slags mening.

### Opgaver

##### Opgave 1.1
Vi ser på, hvad *afstanden* mellem to punkter i det latente rum egentlig fortæller.

Kig på plottet ovenfor. Prøv at forklare, hvorfor det giver god mening, at **hund** og **kat** ligger tæt
på hinanden, men langt fra **bil**.

Se så om du kan sætte ord på, hvad det ville betyde, hvis to ords vektorer lå helt oven i hinanden.

Hint: Tænk på afstanden som "hvor forskellige er de to ting?" — hvad siger en afstand på næsten nul så om
de to ord?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.2
Nu prøver vi selv at tilføje et ord til det latente rum.

Indtil nu har vi kun kigget på de ord, vi startede med. Men et latent rum er ikke fastlåst — vi kan udmærket
placere nye ord i det. Idéen er, at når to ord betyder noget, der ligner hinanden, så skal deres punkter
også ligge tæt på hinanden.

Prøv at give ordet `"prinsesse"` sin egen vektor med to tal, og placér den, så den lander tæt på
**dronning**. Kør så cellen og se, hvor ordet havner på plottet.

Hint: Kig på dronnings vektor i cellen længere oppe — hvad skal dine to tal ligne, for at prinsesse ender
lige ved siden af hende?

In [ ]:
word_vectors["prinsesse"] = torch.tensor([...])   # <-- udfyld: to tal tæt på "dronning", fx [2.2, 2.8]

# Vi tegner alle ordene igen — nu med "prinsesse" iblandt.
plt.figure(figsize=(6, 5))
for word, v in word_vectors.items():
    plt.scatter(v[0], v[1])
    plt.annotate(word, (v[0], v[1]), fontsize=12)
plt.title("Med dit eget ord")
plt.show()   # "prinsesse" skal nu dukke op tæt på "dronning"

# 2: Ord og tegn som vektorer — `nn.Embedding`

Hvor kommer vektorerne fra i en rigtig model? Fra et **opslagsbord** kaldet en *embedding*: hver token (her:
hvert tegn) har et nummer, og embedding-bordet slår nummeret op og giver en vektor. Det er det **allerførste**,
en transformer gør ved din tekst.

In [ ]:
# En embedding-tabel er et opslagsbord: her med plads til 6 tokens, hvor hver token får en vektor med 4 tal.
embedding = nn.Embedding(num_embeddings=6, embedding_dim=4)

token_ids = torch.tensor([0, 3, 5])     # vi vælger tre tokens ud fra deres numre
vectors = embedding(token_ids)          # og slår deres vektorer op i bordet

print("Token-id'er:", token_ids.tolist())
print("Deres vektorer:\n", vectors)
print("Form:", tuple(vectors.shape), "= (3 tokens, hver med 4 tal)")

Lige nu er tallene **tilfældige** — bordet er ikke trænet endnu. Under træning skubbes vektorerne
rundt, indtil "mening" ligger i geometrien. I næste afsnit kigger vi på et bord, der ER færdigtrænet.

### Opgaver

##### Opgave 2.1
Vi bygger selv et embedding-bord — det opslagsbord fra afsnittet ovenfor, der laver et token-nummer om til
en vektor.

Et embedding-bord skal vide to ting, når det oprettes: hvor mange forskellige tokens der er plads til
(`num_embeddings`), og hvor mange tal hver tokens vektor skal have (`embedding_dim`). I koden nedenfor er
det første tal allerede sat til 10 — så mangler du kun at bestemme, hvor lang hver vektor skal være.

Prøv at lave en embedding-tabel med **10 tokens**, hvor hver token bliver til en vektor med **16 tal**.
Slå så token nummer 3 op og kig på formen.

Hint: `embedding_dim` er "hvor mange tal hver token skal have". Kig på eksemplet i cellen ovenfor, hvor
bordet gav 4 tal pr. token.

In [ ]:
my_embedding = nn.Embedding(num_embeddings=10, embedding_dim=...)   # <-- udfyld: hvor mange tal skal hver vektor have?

vector3 = my_embedding(torch.tensor(3))   # slå token nummer 3 op i dit nye bord
print("Vektor for token 3:", vector3)
print("Form:", tuple(vector3.shape))      # skal give (16,)

##### Opgave 2.2
Vi ser på, hvorfor et tegn skal beskrives med en hel *vektor* og ikke med ét enkelt tal.

Et embedding-bord med 84 tegn og 384 tal pr. tegn har 84 × 384 ≈ 32.000 tal, som modellen selv skal lære.
Det er mange tal — så man kunne fristes til at spørge: hvorfor ikke nøjes med tegnets nummer (0, 1, 2, …)
direkte i stedet for en hel vektor?

Prøv at forklare, hvad en hel talrække kan udtrykke, som et enkelt nummer ikke kan.

Hint: Hvis a = 1 og b = 2, er b så "dobbelt så meget værd" som a? Tænk på, hvor mange forskellige retninger
et enkelt tal kan pege i — og hvor mange en vektor kan.

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

# 3: Kig ind i den TRÆNEDE models latente rum

Nu bruger vi ikke et tilfældigt bord — vi henter det **rigtige** embedding-bord ud af den færdige
transformer (`base_model.pth`), som I skal lege med i workshoppen. Modellen arbejder på tegn, og den har
et fast vokabular på 84 tegn.

In [ ]:
# Det FASTE vokabular fra transformeren (samme 84 tegn som i workshoppen).
VOCAB_CHARS = (
    "\n "                            # linjeskift og mellemrum
    "abcdefghijklmnopqrstuvwxyz"     # små bogstaver
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"     # store bogstaver
    "æøåÆØÅ"                          # danske bogstaver
    "0123456789"                     # tal
    ".,!?:;-'\"*()[]"                # tegnsætning
)
chars = sorted(set(VOCAB_CHARS))     # samme rækkefølge som modellen bruger
print("Antal tegn i vokabularet:", len(chars))

# Hent embedding-bordet ud af den trænede model.
state = torch.load("base_model.pth", map_location="cpu")
E = state["transformer.wte.weight"]      # form (84, 384): én vektor pr. tegn
print("Embedding-bord:", tuple(E.shape), "= (84 tegn, 384 tal hver)")

384 tal pr. tegn kan vi ikke plotte direkte. Vi projicerer ned til **2D** med PCA — det beholder de
to "vigtigste" retninger i dataen (samme grundidé som autoencoderens flaskehals).

In [ ]:
# Projicér de 384-dimensionelle vektorer ned til 2 dimensioner med PCA, så vi kan tegne dem.
E_centered = E - E.mean(dim=0)           # træk gennemsnittet fra, så tallene ligger pænt omkring nul
U, S, V = torch.pca_lowrank(E_centered, q=2)
coords = E_centered @ V[:, :2]           # form (84, 2): to koordinater pr. tegn

def nice(ch):
    # Gør linjeskift og mellemrum synlige, så de kan aflæses på plottet.
    return {"\n": "\\n", " ": "␣"}.get(ch, ch)

plt.figure(figsize=(11, 9))
for i, ch in enumerate(chars):
    plt.scatter(coords[i, 0], coords[i, 1], color="steelblue")
    plt.annotate(nice(ch), (coords[i, 0], coords[i, 1]), fontsize=11)
plt.title("Transformerens tegn-embeddings projiceret til 2D")
plt.show()

Uden at nogen har fortalt den det, har modellen samlet tegn, der *opfører sig ens*: bogstaver ét
sted, cifre et andet, tegnsætning for sig. Den lærte det udelukkende ved at gætte det næste tegn millioner
af gange. **Det er dét latente rum, transformeren "tænker" i.**

Vi kan også måle nærhed direkte med **cosine similarity** (måler vinklen mellem to vektorer: 1 = samme
retning/meget ens, 0 = vinkelret, −1 = modsat).

In [ ]:
def cosine_similarity(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm())

# Vi trækker "gennemsnits-tegnet" fra først (samme centrering som i PCA'en ovenfor),
# så vi sammenligner tegnenes RETNING og ikke den fælles baggrund, som alle tegn deler.
E_dir = E - E.mean(dim=0)

def nearest_chars(ch, n=5):
    # Finder de n tegn, hvis retning ligner ch's mest (cosine similarity).
    i = chars.index(ch)
    sims = [(cosine_similarity(E_dir[i], E_dir[j]).item(), other)
            for j, other in enumerate(chars) if j != i]
    sims.sort(reverse=True)
    return [(round(s, 2), c) for s, c in sims[:n]]

print("Tegn der ligner 'a' mest:", nearest_chars("a"))
print("Tegn der ligner 't' mest:", nearest_chars("t"))
print("Tegn der ligner '5' mest:", nearest_chars("5"))

Læs outputtet: **a** ligger tættest på andre **vokaler** (e, o, i), og **t** tæt på andre almindelige
**konsonanter** (r, n, d). Men **5** havner sammen med sjældne tegn som Ø, Æ og q — ikke fordi et 5-tal
"ligner" et ø, men fordi modellen **næsten aldrig har set dem** i fantasy-teksten og derfor ikke har lært
dem ordentligt endnu. Det latente rum afslører altså også, hvad modellen *ikke* har øvet sig på!

### Opgaver

##### Opgave 3.1
Vi bygger cosine similarity selv, så vi kan se, præcis hvordan nærhed måles i det latente rum.

Se om du kan færdiggøre `my_cosine`. Cosine similarity er prikproduktet af de to vektorer **divideret med**
deres længder ganget sammen. Testen nederst sammenligner dit svar med PyTorchs egen — de skal give (næsten)
det samme.

Hint: En vektors længde er `a.norm()`. Du skal altså dividere med `a`'s længde gange `b`'s længde.

In [ ]:
def my_cosine(a, b):
    return torch.dot(a, b) / (...)   # <-- udfyld: divider med a's længde gange b's længde (brug .norm())

# Test — skal give (næsten) det samme som torch's egen:
print("min:  ", round(my_cosine(E[0], E[1]).item(), 4))
print("torch:", round(torch.nn.functional.cosine_similarity(E[0], E[1], dim=0).item(), 4))

##### Opgave 3.2
Nu udforsker vi det trænede latente rum tegn for tegn.

Funktionen `nearest_chars` tager ét tegn og finder de tegn, hvis vektorer peger mest i samme retning — altså
dem, modellen opfatter som mest "ens". I cellen nedenfor står tegnet allerede i variablen `my_char`, så du
skal kun bytte selve tegnet ud og køre cellen igen.

Prøv at skifte tegnet ud og se på dets nærmeste naboer. Passer naboerne med din egen intuition om, hvilke
tegn der hører sammen?

Hint: Prøv både et helt almindeligt bogstav og et sjældent tegn (fx `"7"` eller `"æ"`) — husk mønstret fra
teksten ovenfor, hvor de sjældne tegn klumpede sammen, fordi modellen næsten aldrig har set dem.

In [ ]:
my_char = "q"   # ← prøv fx "æ", "7", "!" eller et mellemrum " "
print(f"Tegn der ligner {my_char!r} mest:", nearest_chars(my_char))

##### Opgave 3.4
Vi samler trådene fra hele notebooken.

Vi har set, at hvert tegn bliver til en vektor i et latent rum, hvor "ens" tegn ligger tæt. Det er kun det
**første** skridt i en transformer — bagefter blander den vektorerne sammen med *attention*.

Prøv at forklare, hvorfor du tror, det er en god start at have tegn med samme rolle placeret tæt på
hinanden, før modellen regner videre.

Hint: Tænk på det latente rum som modellens skrivebord — er der bedre arbejdsro til at bygge videre, når
tingene ligger pænt sorteret, eller når de ligger hulter til bulter?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

## Ekstra opgaver

Her er nogle ekstra udfordringer, hvis du er nået hele vejen igennem og har lyst til mere. De bygger videre
på det, du allerede har lavet.

##### Ekstra 1
Vi zoomer ind på én gruppe tegn ad gangen for at se, om de virkelig klumper sammen i det latente rum.

Cellen plotter lige nu kun **vokalerne**. Kør den og se, om de samler sig i ét område. Prøv så at lave det
samme plot for cifrene `"0123456789"` og sammenlign de to grupper — ligger cifrene også pænt sammen?

Hint: Du behøver kun at ændre listen i den første linje (`vowels = ...`) — resten af plot-koden kan blive
stående, som den er.

In [ ]:
vowels = list("aeiouyæøå")   # <-- prøv bagefter list("0123456789") for cifrene

plt.figure(figsize=(7, 6))
for ch in vowels:
    i = chars.index(ch)                      # find tegnets plads i vokabularet
    plt.scatter(coords[i, 0], coords[i, 1], color="crimson")
    plt.annotate(ch, (coords[i, 0], coords[i, 1]), fontsize=14)
plt.title("Kun vokaler i det latente rum")   # (skift gerne titlen, når du plotter cifrene)
plt.show()